In [11]:
import os, time, psutil
import numpy as np
from scipy import stats

def mem_gb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**3)

def make_data_once(X_path="X.npy", y_path="y.npy", n=2_000_000, p=400, seed=0, dtype=np.float32):
    rng = np.random.default_rng(seed)
    print("[MAKE DATA]")
    print("RSS start (GB):", round(mem_gb(), 3))
    t0 = time.time()
    X = rng.normal(0, 1, size=(n, p)).astype(dtype)
    w_true = rng.normal(0, 1, size=p).astype(dtype)
    eps = rng.normal(0, 0.5, size=n).astype(dtype)
    y = (0.2 + X @ w_true + eps).astype(dtype)
    print("gen time (s):", round(time.time() - t0, 2))
    print("RSS after gen (GB):", round(mem_gb(), 3))
    np.save(X_path, X)
    np.save(y_path, y)
    del X, y
    print("saved:", X_path, y_path)
    print("RSS after save+del (GB):", round(mem_gb(), 3))

def ols_batch_from_file(X_path="X.npy", y_path="y.npy", add_intercept=True, batch_size=1_000_000):
    X = np.load(X_path, mmap_mode="r")   # float32 mmap
    y = np.load(y_path, mmap_mode="r")   # float32 mmap
    n, p = X.shape
    k = p + 1 if add_intercept else p

    print("\n[BATCH from file]")
    print(f"n={n:,}, p={p}, batch={batch_size:,}, intercept={add_intercept}")
    print("RSS start (GB):", round(mem_gb(), 3))

    # pass1
    t0 = time.time()
    XtX = np.zeros((k, k), dtype=np.float64)
    Xty = np.zeros((k,), dtype=np.float64)

    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)
        # >>> 改動1：不轉 float64（避免 3.2GB 拷貝）
        Xb = np.asarray(X[s:e], dtype=np.float32)
        yb = np.asarray(y[s:e], dtype=np.float32)
        m = e - s

        if add_intercept:
            XtX[0,0] += m

            # >>> 改動2：sum 用 float64 累加（小成本、低誤差）
            sx = Xb.sum(axis=0, dtype=np.float64)          # (p,)
            XtX[0,1:] += sx
            XtX[1:,0] += sx

            # >>> 改動3：先算小矩陣/小向量，再轉 float64 累加
            XtX[1:,1:] += (Xb.T @ Xb).astype(np.float64, copy=False)

            Xty[0] += float(yb.sum(dtype=np.float64))
            Xty[1:] += (Xb.T @ yb).astype(np.float64, copy=False)

        print(f"pass1 chunk {s//batch_size+1} | RSS(GB)={mem_gb():.3f}")

    print("pass1 time (s):", round(time.time() - t0, 2))
    print("RSS after pass1 (GB):", round(mem_gb(), 3))

    # solve
    t1 = time.time()
    beta = np.linalg.solve(XtX, Xty)
    print("solve time (s):", round(time.time() - t1, 2))
    print("RSS after solve (GB):", round(mem_gb(), 3))

    # pass2
    t2 = time.time()
    SSE = 0.0
    for s in range(0, n, batch_size):
        e = min(s + batch_size, n)
        # >>> 改動4：pass2 也不轉 float64
        Xb = np.asarray(X[s:e], dtype=np.float32)
        yb = np.asarray(y[s:e], dtype=np.float32)

        if add_intercept:
            # Xb 是 float32，beta 是 float64，yhat 會是 float64
            yhat = beta[0] + Xb @ beta[1:]
        else:
            yhat = Xb @ beta

        # >>> 改動5：只把 yb 以 view/cast 對齊到 float64（不拷貝大 X）
        r = yb.astype(np.float64, copy=False) - yhat
        SSE += float(r @ r)

        print(f"pass2 chunk {s//batch_size+1} | RSS(GB)={mem_gb():.3f}")

    df = n - k
    sigma2 = SSE / df
    cov_beta = sigma2 * np.linalg.inv(XtX)
    se = np.sqrt(np.diag(cov_beta))
    t_stat = beta / se
    p_val = 2.0 * stats.t.sf(np.abs(t_stat), df=df)

    print("pass2(infer) time (s):", round(time.time() - t2, 2))
    print("RSS after pass2 (GB):", round(mem_gb(), 3))
    print("df:", df)
    print("cond(X'X):", float(np.linalg.cond(XtX)))

    return beta, se, t_stat, p_val

if __name__ == "__main__":
    make_data_once(n=2_000_000, p=400, seed=0, dtype=np.float32)
    b, se, t_stat, p_val = ols_batch_from_file(add_intercept=True, batch_size=1_000_000)

    print("\nFirst 10 coefficients (incl intercept):")
    for j in range(10):
        name = "intercept" if j == 0 else f"x{j}"
        print(f"{name:10s} beta={b[j]: .6f}  se={se[j]: .6f}  t={t_stat[j]: .3f}  p={p_val[j]: .3e}")


[MAKE DATA]
RSS start (GB): 0.026
gen time (s): 23.67
RSS after gen (GB): 2.999
saved: X.npy y.npy
RSS after save+del (GB): 0.017

[BATCH from file]
n=2,000,000, p=400, batch=1,000,000, intercept=True
RSS start (GB): 0.01
pass1 chunk 1 | RSS(GB)=1.507
pass1 chunk 2 | RSS(GB)=3.002
pass1 time (s): 2.42
RSS after pass1 (GB): 3.002
solve time (s): 0.08
RSS after solve (GB): 3.003
pass2 chunk 1 | RSS(GB)=3.018
pass2 chunk 2 | RSS(GB)=3.018
pass2(infer) time (s): 2.27
RSS after pass2 (GB): 3.024
df: 1999599
cond(X'X): 1.0573535012169526

First 10 coefficients (incl intercept):
intercept  beta= 0.199806  se= 0.000354  t= 564.927  p= 0.000e+00
x1         beta=-0.388074  se= 0.000353  t=-1098.551  p= 0.000e+00
x2         beta=-1.477563  se= 0.000354  t=-4177.091  p= 0.000e+00
x3         beta=-0.667777  se= 0.000354  t=-1887.572  p= 0.000e+00
x4         beta= 0.485863  se= 0.000354  t= 1373.732  p= 0.000e+00
x5         beta= 0.128018  se= 0.000353  t= 362.205  p= 0.000e+00
x6         beta= 1.18